## PCA 

In [4]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("cmi_internet_pulito.csv")

pca_strategy = {
    'BIA-': 4,       # Aumentato da 2 a 4
    'FGC-': 4,       # Aumentato drasticamente da 1 a 4
    'Physical-': 3   # Aumentato da 2 a 3
}

colonne_da_mantenere = [
    'id', 'sii', 'Basic_Demos-Age', 'Basic_Demos-Sex', 
    'PreInt_EduHx-computerinternet_hoursday', 'CGAS-CGAS_Score',
    'PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total', 'SDS-SDS_Total_T'
]
colonne_stagioni = [col for col in df.columns if 'Season' in col]
colonne_da_mantenere.extend(colonne_stagioni)



df_ridotto = df[colonne_da_mantenere].copy()

In [18]:


# 1. Invece di decidere il numero di colonne, decidi la % di informazione!
pca_strategy = {
    'BIA-': 0.85,       # Voglio l'85% dell'informazione BIA
    'FGC-': 0.80,       # Voglio l'80% dell'informazione motoria (ci accontentiamo per evitare troppe colonne)
    'Physical-': 0.85   # Voglio l'85% dell'informazione vitale
}

# ... (il resto dell'impostazione delle colonne rimane identico) ...

for prefix, target_var in pca_strategy.items():
    
    cols_blocco = [col for col in df.columns if col.startswith(prefix) and 'Season' not in col]
    
    if len(cols_blocco) > 0:
        print(f"\nElaborazione blocco '{prefix}' ({len(cols_blocco)} feature originali)...")
        
        X_blocco = df[cols_blocco]
        
        # Standardizzazione
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_blocco)
        
        # Applicazione della PCA passando il float (target_var)
        pca = PCA(n_components=target_var, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        
        # pca.n_components_ ti dice QUANTE componenti ha dovuto creare per raggiungere il target
        num_componenti_generate = pca.n_components_
        varianza_spiegata = pca.explained_variance_ratio_.sum() * 100
        
        print(f" -> Generate {num_componenti_generate} componenti per spiegare il {varianza_spiegata:.2f}% della varianza.")
        
        # Aggiungiamo dinamicamente le nuove colonne al df_ridotto
        for i in range(num_componenti_generate):
            nome_nuova_colonna = f"{prefix.strip('-')}_PC{i+1}"
            df_ridotto[nome_nuova_colonna] = X_pca[:, i]


Elaborazione blocco 'BIA-' (16 feature originali)...
 -> Generate 10 componenti per spiegare il 86.89% della varianza.

Elaborazione blocco 'FGC-' (14 feature originali)...
 -> Generate 8 componenti per spiegare il 80.27% della varianza.

Elaborazione blocco 'Physical-' (7 feature originali)...
 -> Generate 4 componenti per spiegare il 85.08% della varianza.


In [19]:
df_ridotto.head()

,id,sii,Basic_Demos-Age,Basic_Demos-Sex,PreInt_EduHx-computerinternet_hoursday,CGAS-CGAS_Score,PAQ_A-PAQ_A_Total,PAQ_C-PAQ_C_Total,SDS-SDS_Total_T,Basic_Demos-Enroll_Season,...,FGC_PC3,FGC_PC4,FGC_PC5,FGC_PC6,FGC_PC7,FGC_PC8,Physical_PC1,Physical_PC2,Physical_PC3,Physical_PC4
0,0,2.0,5,0.0,3.0,51.00,1.83,2.83,57.12,Fall,...,0.884270,0.122905,-0.359161,-0.416067,-1.929473,-0.617682,-1.371182,-0.569826,-0.149656,-1.099046
1,1,0.0,9,0.0,0.0,65.48,2.01,2.34,64.00,Summer,...,-0.828741,-2.324641,-2.508093,-0.616961,1.100989,-0.120572,-1.611008,0.719974,-0.950126,-1.696749
2,2,0.0,10,1.0,2.0,71.00,2.00,2.17,54.00,Summer,...,-4.185658,-1.450839,-1.842050,0.472904,1.229945,0.194158,-0.538302,0.479503,0.568388,0.679833
3,3,1.0,9,0.0,0.0,71.00,2.01,2.45,45.00,Winter,...,0.023204,-0.156512,0.718484,1.653427,-0.116547,-0.519408,-0.458001,0.282267,1.095267,0.822475
4,4,0.0,18,1.0,1.0,65.00,1.04,2.52,56.28,Spring,...,0.604250,0.694965,0.188695,-0.233722,0.444991,0.598329,-0.170494,0.367067,-0.172337,0.112118
